# 09주 · Manning 등류·원형 개수로

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 등류와 Manning 식 / 최적 단면·부분 만관 흐름

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week09/CE272_W09.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week09) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: Manning·원형 수로의 1도 격자

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 원본 사다리꼴 등류 예제

**슬라이드 연결**: CE272_W09a_Uniform_Flow_1.pptx p11, 12, 13, 14 / CE272_W09b_Uniform_Flow_2.pptx p3, 7, 11, 13

**자료 구분**: 원본 예제 6.5

**먼저 예측**: 같은 단면적이면 윤변이 달라도 유량은 같은가?

**문제**: 바닥 폭 2 m, 수심 1 m, 측면경사 2:1인 수로의 유속과 유량을 구하시오. Manning n=0.012, 바닥 경사 0.001을 사용하시오. 원본의 Rh=0.62 반올림 계산과 비교하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
y,b,z,n,S0 = 1.,2.,2.,.012,.001
s=hyd.section(y,b,z);Q=hyd.manning(y,b,z,n,S0)
pd.Series({**s,'V_m_s':Q/s['A'],'Q_m3_s':Q,
           'rounded_R_0_62_Q_m3_s':s['A']/n*.62**(2/3)*np.sqrt(S0)})

**검토 문제**: 원본 표시값 7.664 m³/s와 정밀 기하 계산의 차이를 설명하시오. 조도나 경사를 바꾼 실험과 반올림 차이를 구분하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 원형 수로의 최대 유량과 최대 유속

**슬라이드 연결**: CE272_W09b_Uniform_Flow_2.pptx p20, 21, 22, 23, 24, 25

**자료 구분**: 원본 Excel·Colab 원리 재구성

**먼저 예측**: 자유수면을 유지하는 원형 수로의 최대 유량은 만관에서 발생하는가?

**문제**: 같은 Manning n과 에너지경사에서 원형 수로의 Q/Qfull, V/Vfull을 중심각 θ의 함수로 계산하시오. 최대 유량과 최대 유속의 수심비가 다른 이유를 설명하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
theta=np.linspace(.01,2*np.pi,2001)
Qratio,Vratio,yD=hyd.circular_ratios(theta)
display(hyd.circular_maxima())
plt.plot(yD,Qratio,label='Q/Qfull');plt.plot(yD,Vratio,label='V/Vfull')
plt.axhline(1,color='gray',linestyle='--');plt.xlabel('y/D');plt.legend();plt.grid();plt.show()
print('Full-flow limit:',hyd.circular_ratios(2*np.pi))

**검토 문제**: 수치미분으로 구한 원본 Colab의 최대점과 해석 미분식의 근을 비교하시오. 압력 관수로·퇴적·조도 변화가 생기면 이 곡선을 그대로 사용할 수 있는지 검토하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

단면·조도·경사의 영향과 최대 유량·최대 유속 조건을 구분하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')